In [ ]:

# 1. INSTALL DEPENDENCIES
!pip install tree-sitter==0.21.3 tree-sitter-languages==1.10.2


In [ ]:
# ==============================================================================
#  UNIXCODER PIPELINE (Phase 1 -> 2 -> 3)
#  Features: Canonicalization, Hard Negative Mining, Latest & Best Saving
# ==============================================================================

import os
import gc
import json
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import tree_sitter_languages
from torch.utils.data import WeightedRandomSampler, DataLoader
from datasets import Dataset, concatenate_datasets
from transformers import (
    RobertaTokenizer, RobertaModel, RobertaPreTrainedModel,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import drive
import warnings

warnings.filterwarnings("ignore")

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')

# Model & Paths
MODEL_CHECKPOINT = "microsoft/graphcodebert-base"
FOLDER_NAME = "Run_GCB"
SUB_PREFIX = "submission_GCB"

# Hyperparameters
FIXED_LEARNING_RATE = 3e-5
MAX_LENGTH = 512
BATCH_SIZE = 128

# Steps
LOGGING_STEPS = 2000
EVAL_STEPS = 2000
SAVE_STEPS = 2000

# Directories
BASE_DIR = "/content/drive/MyDrive/SemEval_Models"
JSON_PATH = "/content/hard_negative_indices.json"
OUTPUT_ROOT = os.path.join(BASE_DIR, FOLDER_NAME)

os.makedirs(BASE_DIR, exist_ok=True)

print(f"🚀 Starting {MODEL_CHECKPOINT} | Max Len: {MAX_LENGTH} | Batch: {BATCH_SIZE} | LR: {FIXED_LEARNING_RATE}")

# --- 2. ADVANCED CANONICALIZER ---
class UniversalCanonicalizer:
    def __init__(self):
        self.lang_map = {
            'python': 'python', 'py': 'python', 'java': 'java', 'cpp': 'cpp',
            'c++': 'cpp', 'c': 'c', 'c#': 'c_sharp', 'cs': 'c_sharp',
            'javascript': 'javascript', 'js': 'javascript', 'php': 'php', 'go': 'go'
        }
        self.parsers = {}

    def _get_parser(self, lang_key):
        parser_name = self.lang_map.get(lang_key, 'python')
        if parser_name in self.parsers: return self.parsers[parser_name]
        try:
            parser = tree_sitter_languages.get_parser(parser_name)
            self.parsers[parser_name] = parser
            return parser
        except: return None

    def canonicalize(self, code_str, lang_label):
        if not isinstance(code_str, str): return ""
        lang_key = str(lang_label).lower()
        parser = self._get_parser(lang_key)
        if not parser: return code_str

        code_bytes = code_str.encode('utf8')
        try:
            tree = parser.parse(code_bytes)
            code_bytes = self._rename_identifiers(code_bytes, tree.root_node)
            canon_code = code_bytes.decode('utf8', errors='ignore')
            return self._flatten_layout(canon_code, self.lang_map.get(lang_key, 'python'))
        except: return code_str

    def _rename_identifiers(self, code_bytes, root_node):
        replacements = []
        counters = {'VAR': 0, 'FUNC': 0, 'TYPE': 0}
        name_map = {}
        keywords = {'if', 'else', 'for', 'while', 'return', 'class', 'def', 'void', 'int', 'float', 'double', 'string', 'public', 'private', 'static', 'import', 'from', 'include', 'var', 'let', 'const', 'try', 'catch', 'true', 'false', 'null'}

        def traverse(node):
            is_id = ("identifier" in node.type or "variable" in node.type or "name" in node.type)
            if is_id and len(node.children) == 0:
                original_name = code_bytes[node.start_byte:node.end_byte].decode('utf8', errors='ignore')
                if original_name not in keywords and len(original_name) > 1:
                    parent_type = node.parent.type if node.parent else ""
                    category = 'VAR'
                    if ('function' in parent_type or 'method' in parent_type or 'call' in parent_type): category = 'FUNC'
                    elif ('class' in parent_type or 'type' in parent_type): category = 'TYPE'

                    if original_name not in name_map:
                        name_map[original_name] = f"{category}_{counters[category]}"
                        counters[category] += 1
                    replacements.append((node.start_byte, node.end_byte, name_map[original_name]))
            for child in node.children: traverse(child)

        traverse(root_node)
        replacements.sort(key=lambda x: x[0], reverse=True)
        mutable_code = bytearray(code_bytes)
        for start, end, new_text in replacements: mutable_code[start:end] = new_text.encode('utf8')
        return mutable_code

    def _flatten_layout(self, code_str, lang_type):
        lines = code_str.split('\n')
        processed_lines = []
        comment_pattern = re.compile(r'//.*|#.*|/\*.*?\*/', re.DOTALL)
        for line in lines:
            line = re.sub(comment_pattern, '', line)
            line = line.replace('\t', '    ').rstrip() if 'python' in lang_type else line.strip()
            if line: processed_lines.append(line)
        return "\n".join(processed_lines)

# --- 3. AUGMENTATION PIPELINE ---
class AugmentationPipeline:
    def __init__(self, aug_prob_hard=1.0, aug_prob_easy=0.5):
        self.engine = UniversalCanonicalizer()
        self.aug_prob_hard = aug_prob_hard
        self.aug_prob_easy = aug_prob_easy

    def process_batch(self, examples):
        out_codes, out_labels, out_weights = [], [], []
        batch_size = len(examples['code'])
        for i in range(batch_size):
            orig, label = examples['code'][i], examples['label'][i]
            is_hard = examples['is_hard'][i] if 'is_hard' in examples else False
            lang = examples['language'][i] if 'language' in examples else 'python'

            weight = 5.0 if is_hard else (3.0 if label > 0 else 1.0)

            out_codes.append(orig)
            out_labels.append(label)
            out_weights.append(weight)

            threshold = self.aug_prob_hard if is_hard else self.aug_prob_easy
            if np.random.random() < threshold:
                canon = self.engine.canonicalize(orig, lang)
                if canon and len(canon) > 5:
                    out_codes.append(canon)
                    out_labels.append(label)
                    out_weights.append(weight)

        return {"code": out_codes, "label": out_labels, "weight": out_weights}

# --- 4. CUSTOM MODEL ---
class CustomUnixCoder(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size * 2, config.num_labels)
        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        kwargs.pop("num_items_in_batch", None)
        outputs = self.roberta(input_ids, attention_mask=attention_mask, **kwargs)
        seq_out = outputs[0]

        mask = attention_mask.unsqueeze(-1).expand(seq_out.size()).float()
        mean_p = torch.sum(seq_out * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

        seq_out_copy = seq_out.clone()
        seq_out_copy[attention_mask == 0] = -1e9
        max_p, _ = torch.max(seq_out_copy, 1)

        logits = self.classifier(self.dropout(torch.cat((mean_p, max_p), 1)))
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return (loss, logits) if loss is not None else logits

class WeightedTrainer(Trainer):
    def __init__(self, *args, custom_sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.custom_sampler = custom_sampler

    def get_train_dataloader(self):
        if self.custom_sampler is None: return super().get_train_dataloader()
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=self.custom_sampler,
            collate_fn=self.data_collator,
            num_workers=4,
            pin_memory=True
        )

# --- 5. DATA LOADING (FIXED) ---
print("[INFO] Loading Tokenizer & Data...")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_CHECKPOINT)

train_df = pd.read_parquet(os.path.join(BASE_DIR, "train.parquet")).dropna(subset=['code', 'label']).reset_index(drop=True)
val_df = pd.read_parquet(os.path.join(BASE_DIR, "validation.parquet")).dropna(subset=['code', 'label']).reset_index(drop=True)
test_df = pd.read_parquet(os.path.join(BASE_DIR, "test.parquet"))

print(f"[INFO] Using FULL Validation Set: {len(val_df)} samples")

if os.path.exists(JSON_PATH):
    print(f"[INFO] Hard negative JSON found at {JSON_PATH}")
    with open(JSON_PATH, "r") as f:
        hard_indices = set(json.load(f)["hard_indices"])
    train_df['is_hard'] = train_df.index.isin(hard_indices)
else:
    print(f"[INFO] Couldn't find Hard negative JSON at {JSON_PATH}")
    train_df['is_hard'] = False

if 'language' not in train_df.columns: train_df['language'] = 'python'

print("[INFO] Augmenting TRAIN Data...")
aug_pipeline = AugmentationPipeline()

# Orijinal dataframe'deki sütunları saklayıp augmentasyondan sonra temizlemek için
columns_to_remove_pre = train_df.columns.tolist()

ds_train_raw = Dataset.from_pandas(train_df).map(
    aug_pipeline.process_batch,
    batched=True,
    batch_size=1000,
    num_proc=4,
    remove_columns=columns_to_remove_pre
)

def tokenize_fn(x):
    return tokenizer(x["code"], truncation=True, max_length=MAX_LENGTH)

print("[INFO] Tokenizing Datasets...")

# !!! HATA DÜZELTME: Tokenize ederken 'code' ve 'weight' sütunlarını siliyoruz !!!
# Trainer 'weight' veya 'code' sütununu görünce hata verir.
cols_to_remove_train = ["code", "weight"] if "weight" in ds_train_raw.column_names else ["code"]

ds_train = ds_train_raw.map(
    tokenize_fn,
    batched=True,
    num_proc=4,
    remove_columns=cols_to_remove_train # <-- KRİTİK DÜZELTME BURADA
)

# Validation için de aynısı
val_cols_remove = [c for c in val_df.columns if c != 'label']
ds_val = Dataset.from_pandas(val_df).map(
    tokenize_fn,
    batched=True,
    num_proc=4,
    remove_columns=val_cols_remove
)
ds_full_val = ds_val

# Test için de aynısı
test_cols_remove = [c for c in test_df.columns if c != 'label']
ds_test = Dataset.from_pandas(test_df).map(
    tokenize_fn,
    batched=True,
    num_proc=4,
    remove_columns=test_cols_remove
)

# PyTorch için sütun isimlendirmesi
for ds in [ds_train, ds_val, ds_full_val, ds_test]:
    if "label" in ds.column_names: ds.rename_column("label", "labels")
    # Formatı torch yapıyoruz ki gereksiz stringler tamamen gitsin
    ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --- 6. TRAINING HELPER FUNCTIONS ---
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    _, _, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='macro', zero_division=0)
    return {'accuracy': accuracy_score(p.label_ids, preds), 'f1_macro': f1}

def save_res(model, ds, df, name, save_probs=True):
    print(f"[INFO] Generating predictions for {name}...")
    temp_trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir="./tmp", per_device_eval_batch_size=BATCH_SIZE, fp16=True, report_to="none"),
        data_collator=data_collator
    )
    out = temp_trainer.predict(ds)

    if save_probs:
        probs_path = os.path.join(BASE_DIR, f"{name}_probs.npy")
        np.save(probs_path, torch.softmax(torch.tensor(out.predictions), dim=-1).numpy())
        print(f"   -> Saved probs to {probs_path}")

    preds = np.argmax(out.predictions, axis=1)
    csv_path = os.path.join(BASE_DIR, f"{name}.csv")
    pd.DataFrame({'ID': df.get('ID', df.index), 'label': preds}).to_csv(csv_path, index=False)
    print(f"   -> Saved CSV to {csv_path}")

def train_and_save_both(trainer, phase_name, sub_prefix):
    print(f"\n▶ Starting Training for {phase_name}...")
    trainer.train()

    print(f"\n💾 Saving LATEST model for {phase_name}...")
    save_res(trainer.model, ds_test, test_df, f"{sub_prefix}_{phase_name}_Latest", save_probs=False)

    best_ckpt = trainer.state.best_model_checkpoint
    if best_ckpt:
        print(f"\n🔄 Loading BEST model checkpoint from: {best_ckpt}")
        best_model = CustomUnixCoder.from_pretrained(best_ckpt, num_labels=11).to(trainer.args.device)
        print(f"💾 Saving BEST model for {phase_name}...")
        save_res(best_model, ds_test, test_df, f"{sub_prefix}_{phase_name}_Best", save_probs=(phase_name == "P2"))
        return best_model
    else:
        print("⚠️ No best checkpoint found. Using latest as best.")
        save_res(trainer.model, ds_test, test_df, f"{sub_prefix}_{phase_name}_Best", save_probs=(phase_name == "P2"))
        return trainer.model


In [ ]:

# --- 7. TRAINING PHASES ---

# PHASE 1: Weighted
print(f"\n🔹 PHASE 1: Weighted Training (LR={FIXED_LEARNING_RATE})...")
model = CustomUnixCoder.from_pretrained(MODEL_CHECKPOINT, num_labels=11)
sampler = WeightedRandomSampler(weights=torch.DoubleTensor(ds_train_raw['weight']), num_samples=len(ds_train_raw), replacement=True)

args_p1 = TrainingArguments(
    output_dir=os.path.join(OUTPUT_ROOT, "p1"),
    num_train_epochs=2,
    learning_rate=FIXED_LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_strategy="steps", save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    load_best_model_at_end=False,
    metric_for_best_model="f1_macro",
    save_total_limit=2,
    fp16=True, report_to="none"
)

trainer1 = WeightedTrainer(
    model=model, args=args_p1, train_dataset=ds_train, eval_dataset=ds_val,
    compute_metrics=compute_metrics, custom_sampler=sampler, data_collator=data_collator
)
model = train_and_save_both(trainer1, "P1", SUB_PREFIX)


In [ ]:

# PHASE 2: Natural
print(f"\n🔹 PHASE 2: Natural Training (LR={FIXED_LEARNING_RATE})...")
args_p2 = TrainingArguments(
    output_dir=os.path.join(OUTPUT_ROOT, "p2"),
    num_train_epochs=4,
    learning_rate=FIXED_LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_strategy="steps", save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    load_best_model_at_end=False,
    metric_for_best_model="f1_macro",
    save_total_limit=2,
    fp16=True, report_to="none"
)

trainer2 = Trainer(
    model=model, args=args_p2, train_dataset=ds_train, eval_dataset=ds_val,
    compute_metrics=compute_metrics, data_collator=data_collator
)
model = train_and_save_both(trainer2, "P2", SUB_PREFIX)


In [ ]:

# PHASE 3: Full
print(f"\n🔹 PHASE 3: Full Training (LR={FIXED_LEARNING_RATE})...")
combined_ds = concatenate_datasets([ds_train, ds_full_val]).shuffle(seed=42)

args_p3 = TrainingArguments(
    output_dir=os.path.join(OUTPUT_ROOT, "p3"),
    num_train_epochs=1,
    learning_rate=FIXED_LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    save_strategy="no", logging_steps=LOGGING_STEPS,
    fp16=True, report_to="none"
)

trainer3 = Trainer(model=model, args=args_p3, train_dataset=combined_ds, data_collator=data_collator)
trainer3.train()

print(f"\n💾 Saving Final Phase 3 Results...")
save_res(model, ds_test, test_df, f"{SUB_PREFIX}_P3_Final", save_probs=True)

print("\n✅ All Phases Completed Successfully!")